## Zoning tests

##### 1 - Draw a sample for checking zones by hand

Draws emails from `email_zones.parquet` for checking zones by hand. With `test_round = False`, it draws 80 emails: 20 at random, then 20 each from emails with banner lines, emails with quoted text, and emails where `corpus.ipynb` block 5 changed a Chipmunk zone, each drawn from emails not already in the sample. Rule 5 and the attachment and meeting-invitation lines in block 5 come from these 80 emails. With `test_round = True`, it draws `n_test` emails at random from outside the 80, to measure the accuracy of the zones; `n_test` rises in steps of 20 until the lower end of the 95% interval for precision of the authored zone reaches 95%.

Files take the prefix `zone_check` or `heldout_check`. Each draw is saved to `_emails.csv` and reused, so rerunning block 5 cannot change it. Saves the drawn emails' lines to `_sample.csv` and writes a labelling tool to `_tool.html`. The tool shows each email without Chipmunk's labels or the zones; the coder marks where each zone starts and exports a CSV, saved as `_coder1.csv`. Compares the coder's zones with Chipmunk's and with the zones in `email_zones.parquet`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Repository root (the folder with _quarto.yml), found from this notebook's folder
root = Path.cwd()
while not (root / '_quarto.yml').exists():
    root = root.parent
here = root / 'corpus' / 'tests' / 'zoning'

# ---- Settings ----
# test_round = False: the 80 emails labelled first, which were then used to refine corpus.ipynb block 5's rules.
# test_round = True: new random emails that corpus.ipynb block 5's rules have never seen, for a clean accuracy figure.
test_round = True
n_test = 40            # emails in the test round; raise by 20 if the stopping rule below says so
zack_dir = str(here)
zones_path = str(root / 'corpus' / 'email_zones.parquet')
prefix = 'heldout_check' if test_round else 'zone_check'
sample_path = os.path.join(zack_dir, f'{prefix}_sample.csv')
tool_path = os.path.join(zack_dir, f'{prefix}_tool.html')
emails_path = os.path.join(zack_dir, f'{prefix}_emails.csv')   # the drawn emails, frozen after the first run
dev_emails_path = os.path.join(zack_dir, 'zone_check_emails.csv')
n_per_stratum = 20
seed = 42

# ---- Emails and what their zones contain ----
z = pd.read_parquet(zones_path)
z = z.sort_values(['msgid2', 'start']).reset_index(drop=True)
z['text'] = z['text'].fillna('').astype(str)
nonblank = z['text'].str.strip() != ''
per_email = (z[nonblank].assign(changed=lambda d: d['zone'] != d['chipmunk_mapped'])
             .groupby('msgid2')
             .agg(banner=('zone', lambda s: (s == 'banner').any()),
                  quoted=('zone', lambda s: (s == 'quoted').any()),
                  changed=('changed', 'any')))

# ---- Draw the sample: random first, then each stratum from emails not yet drawn ----
# The draw depends on corpus.ipynb block 5's zones, so it is saved once and reused; rerunning corpus.ipynb block 5 cannot change it.
strata = [('random', per_email.index.to_series()),
          ('banner', per_email.index[per_email['banner']].to_series()),
          ('quoted', per_email.index[per_email['quoted']].to_series()),
          ('changed', per_email.index[per_email['changed']].to_series())]
drawn = {}
for name, pool_ids in strata:
    left = pool_ids[~pool_ids.isin(list(drawn))]
    for m in left.sample(min(len(left), n_per_stratum), random_state=seed):
        drawn[m] = name
coder1_path = os.path.join(zack_dir, 'zone_check_coder1.csv')
if test_round:
    # New random emails, excluding the 80 already labelled; extra draws keep the emails drawn before
    dev_ids = set(pd.read_csv(dev_emails_path, dtype={'msgid2': str})['msgid2'])
    drawn = {}
    if os.path.exists(emails_path):
        saved = pd.read_csv(emails_path, dtype={'msgid2': str})
        drawn = dict(zip(saved['msgid2'], saved['stratum']))
    need = n_test - len(drawn)
    if need > 0:
        pool_ids = pd.Series(sorted(set(per_email.index.astype(str)) - dev_ids - set(drawn)))
        for m in pool_ids.sample(need, random_state=seed + len(drawn)):
            drawn[m] = 'test'
    print(f"Test emails: {len(drawn)} (none of the {len(dev_ids)} emails used to refine corpus.ipynb block 5)")
elif os.path.exists(emails_path):
    saved = pd.read_csv(emails_path, dtype={'msgid2': str})
    drawn = dict(zip(saved['msgid2'], saved['stratum']))
    print(f"Reusing the saved sample in {emails_path}")
elif os.path.exists(coder1_path):
    # First sample, labelled before the draw was saved: rebuild it from the labels. Its random stratum
    # does not depend on the zones; the other strata were drawn from earlier zones and are marked 'targeted'.
    labelled_ids = pd.read_csv(coder1_path, dtype={'msgid2': str})['msgid2'].unique()
    random_ids = {str(m) for m, s in drawn.items() if s == 'random'}
    drawn = {m: ('random' if m in random_ids else 'targeted') for m in labelled_ids}
    print(f"Rebuilt the sample from the labels in {coder1_path}")
else:
    print("Emails available per stratum:", {name: len(ids) for name, ids in strata})
    drawn = {str(m): s for m, s in drawn.items()}
pd.DataFrame({'msgid2': list(drawn), 'stratum': list(drawn.values())}).to_csv(emails_path, index=False)
print("Emails per stratum:", pd.Series(drawn).value_counts().to_dict())

z['msgid2'] = z['msgid2'].astype(str)
sample = z[z['msgid2'].isin(list(drawn))][['msgid2', 'line', 'start', 'end', 'text']].copy()
sample.insert(0, 'stratum', sample['msgid2'].map(drawn))
sample['manual'] = ''
sample.to_csv(sample_path, index=False)
print(f"Sample: {sample['msgid2'].nunique()} emails, {len(sample):,} lines "
      f"({(sample['text'].str.strip() != '').sum():,} non-blank), saved to: {sample_path}")

# ---- Labelling tool: one offline page; Chipmunk's zones are not included ----
emails = [{'msgid2': str(m), 'lines': g['text'].tolist()}
          for m, g in sample.groupby('msgid2', sort=False)]
order = pd.Series(range(len(emails))).sample(frac=1, random_state=seed).tolist()
emails = [emails[i] for i in order]   # shuffled, so strata are not labelled in blocks

zone_html = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Zone check</title>
<style>
:root { --bg:#f7f7f5; --card:#fff; --ink:#1f2328; --muted:#6b7075; --line:#e2e2de; --sel:#4b5fc4;
  --authored:#2f8f5b; --signature:#8a5cc4; --banner:#c2453d; --quoted:#b7871a; --markup:#4b8fc4; --other:#7a7f85; }
@media (prefers-color-scheme: dark) { :root { --bg:#17181a; --card:#212327; --ink:#e8e8e6; --muted:#9a9ea3; --line:#33363b; --sel:#8b9cf0; } }
* { box-sizing:border-box; }
body { margin:0; background:var(--bg); color:var(--ink); font:15px/1.45 system-ui, -apple-system, sans-serif; }
header { display:flex; flex-wrap:wrap; gap:10px; align-items:center; padding:10px 16px; border-bottom:1px solid var(--line); background:var(--card); position:sticky; top:0; z-index:2; }
header h1 { font-size:16px; margin:0 8px 0 0; }
select, input[type=text], button { font:inherit; font-size:14px; padding:4px 9px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); }
button { cursor:pointer; } button.primary { background:var(--sel); color:#fff; border-color:var(--sel); }
.spacer { flex:1; }
.wrap { display:grid; grid-template-columns:minmax(0,1fr) 320px; gap:16px; max-width:1360px; margin:16px auto; padding:0 16px; }
main { min-width:0; }
aside { position:sticky; top:64px; align-self:start; background:var(--card); border:1px solid var(--line); border-radius:10px; padding:12px 14px; font-size:13px; }
aside .z { margin:0 0 10px; }
aside .chip { display:inline-block; padding:1px 8px; border-radius:999px; color:#fff; font-weight:600; margin-bottom:2px; }
aside .keys { border-top:1px solid var(--line); padding-top:8px; color:var(--muted); line-height:1.9; }
@media (max-width:900px) { .wrap { grid-template-columns:1fr; } aside { position:static; } }
.meta { color:var(--muted); font-size:13px; margin-bottom:8px; display:flex; gap:12px; flex-wrap:wrap; }
.email { background:var(--card); border:1px solid var(--line); border-radius:10px; overflow:hidden; }
.row { display:grid; grid-template-columns:86px 1fr; border-top:1px solid var(--line); cursor:pointer; }
.row:first-child { border-top:0; }
.lab { font-size:12px; padding:3px 8px; color:#fff; display:flex; align-items:center; gap:4px; }
.lab.none { color:var(--muted); background:transparent; }
.lab .start { font-weight:700; }
.txt { padding:3px 10px; font-family:ui-monospace, Menlo, monospace; font-size:13px; white-space:pre-wrap; word-break:break-word; min-height:22px; }
.row.blank .txt { color:var(--muted); }
.row.sel { outline:2px solid var(--sel); outline-offset:-2px; }
.note { width:100%; margin-top:10px; }
.nav { display:flex; justify-content:space-between; align-items:center; margin:12px 0; color:var(--muted); }
#about { max-width:1360px; margin:12px auto 0; padding:0 16px; font-size:14px; }
#about .box { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:14px 18px; }
#about[hidden] { display:none; }
#about td { padding:2px 12px 2px 0; vertical-align:top; }
kbd { border:1px solid var(--line); border-bottom-width:2px; border-radius:4px; padding:0 5px; font-size:12px; background:var(--card); }
.empty { text-align:center; color:var(--muted); padding:40px; }
</style></head><body>
<header>
  <h1>Zone check</h1>
  <button id="aboutBtn">About</button>
  <label>Coder <select id="coder"><option value="">choose…</option><option>coder1</option><option>coder2</option></select></label>
  <span class="spacer"></span><span id="count"></span>
  <button id="import">Import</button><input type="file" id="file" accept=".csv" hidden>
  <button id="export" class="primary">Export CSV</button>
</header>
<section id="about" hidden><div class="box">
  <p>Label the zone of every non-blank line in each email. Zones come in runs, so you only mark where each run starts: select the first line of a run and press its number. Every line below takes that zone until the next mark. A single line inside another zone (a URL inside authored text, for example) is its own run: mark it, then mark the line after it.</p>
  <p>The zone definitions and keys are in the panel on the right. A line holding only punctuation (a lone full stop) takes the zone of the run it falls in; do not mark it. Blank lines need no label.</p>
  <p>Label alone, without seeing the other coder's file. Codes save in this browser for each coder; <b>Export CSV</b> writes <code>__PREFIX___coder1.csv</code> or <code>__PREFIX___coder2.csv</code>. Save it in <code>corpus/tests/zoning</code>. <b>Import</b> reloads an exported file. The page contains email text: keep it and the exported files on storage approved for the study data.</p>
</div></section>
<div class="wrap"><main id="main"></main><aside id="side"></aside></div>
<script id="data" type="application/json">__DATA__</script>
<script>
const emails = JSON.parse(document.getElementById('data').textContent);
const ZONES = [
  ['authored', 'Text the sender wrote for this message, including greetings, closings and a name typed as the sign-off (a name alone, or "Best, Ahmed")'],
  ['signature', 'Signature blocks: names, titles, organizations, addresses, contact details. A name followed by a title or contact details starts the signature. The sender\'s signature placed below a quoted message is still signature'],
  ['banner', 'Inserted or standing notices, wherever they appear, including inside a signature: external-sender warnings, confidentiality disclaimers, work-from-home and remote-work notices, land acknowledgements'],
  ['quoted', 'Earlier messages below a reply or forward header, including the header and the separator line (____ or -----Original Message-----) that opens it'],
  ['markup', 'A line that is only markup, also inside a signature: angle-bracket tags, image references, "image removed by sender", URLs, Outlook Web Access link labels. A line mixing an image and text takes the zone of the text'],
  ['other', 'Separators that open no quoted message, tables, attachment lists, meeting-invitation details, code and log lines']];
const $ = id => document.getElementById(id);
$('side').innerHTML = ZONES.map(([z, d], i) => `<div class="z"><span class="chip" style="background:var(--${z})">${i + 1} ${z}</span><div>${d}</div></div>`).join('') +
  '<div class="keys"><kbd>↑</kbd> <kbd>↓</kbd> select line<br><kbd>1</kbd>–<kbd>6</kbd> start a zone at the selected line<br><kbd>0</kbd> remove the mark<br><kbd>←</kbd> <kbd>→</kbd> previous or next email<br><kbd>N</kbd> note</div>';
$('aboutBtn').onclick = () => { $('about').hidden = !$('about').hidden; };
let coder = '', marks = {}, notes = {}, cur = 0, sel = 0;
const key = () => '__PREFIX__-' + coder;
function load() { try { const s = JSON.parse(localStorage.getItem(key())) || {}; marks = s.marks || {}; notes = s.notes || {}; cur = s.cur || 0; } catch (e) { marks = {}; notes = {}; cur = 0; } sel = 0; }
function save() { try { localStorage.setItem(key(), JSON.stringify({marks, notes, cur})); } catch (e) {} }
function esc(s) { return String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c])); }
function labels(e) {
  const m = marks[e.msgid2] || {}; let z = ''; return e.lines.map((_, i) => (z = m[i] ?? z));
}
function complete(e) { const l = labels(e); return e.lines.every((t, i) => !t.trim() || l[i]); }
function render() {
  const done = emails.filter(complete).length;
  $('count').textContent = `${done} / ${emails.length} emails labelled`;
  if (!coder) { $('main').innerHTML = '<div class="empty">Choose a coder to start. Open About for instructions.</div>'; return; }
  const e = emails[cur], l = labels(e), m = marks[e.msgid2] || {};
  $('main').innerHTML = `
    <div class="meta"><span>Email ${cur + 1} of ${emails.length}</span><span>msgid2 ${esc(e.msgid2)}</span><span>${complete(e) ? 'labelled' : 'not yet labelled'}</span></div>
    <div class="email">${e.lines.map((t, i) => `
      <div class="row ${t.trim() ? '' : 'blank'} ${i === sel ? 'sel' : ''}" data-i="${i}">
        <div class="lab ${l[i] ? '' : 'none'}" style="${l[i] ? `background:var(--${l[i]})` : ''}">${m[i] !== undefined ? '<span class="start">▸</span>' : ''}${l[i] || '—'}</div>
        <div class="txt">${esc(t) || '·'}</div></div>`).join('')}</div>
    <input class="note" id="note" type="text" placeholder="Note on this email (optional)" value="${esc(notes[e.msgid2])}">
    <div class="nav"><button id="prev">← previous</button><button id="next">next →</button></div>`;
  document.querySelectorAll('.row').forEach(r => r.onclick = () => { sel = +r.dataset.i; render(); });
  $('prev').onclick = () => go(-1); $('next').onclick = () => go(1);
  $('note').oninput = ev => { notes[e.msgid2] = ev.target.value; save(); };
  const s = document.querySelector('.row.sel'); if (s) s.scrollIntoView({block: 'nearest'});
}
function go(d) { cur = Math.min(Math.max(cur + d, 0), emails.length - 1); sel = 0; save(); render(); }
function mark(z) { const id = emails[cur].msgid2; marks[id] = marks[id] || {}; if (z) marks[id][sel] = z; else delete marks[id][sel]; save(); render(); }
document.addEventListener('keydown', ev => {
  if (!coder) return;
  if (ev.target.id === 'note') { if (ev.key === 'Escape' || ev.key === 'Enter') ev.target.blur(); return; }
  if (ev.target.tagName === 'SELECT' || ev.target.tagName === 'INPUT') return;
  const n = emails[cur].lines.length;
  if (ev.key === 'ArrowDown') { sel = Math.min(sel + 1, n - 1); ev.preventDefault(); render(); }
  else if (ev.key === 'ArrowUp') { sel = Math.max(sel - 1, 0); ev.preventDefault(); render(); }
  else if (ev.key === 'ArrowRight') go(1);
  else if (ev.key === 'ArrowLeft') go(-1);
  else if (ev.key >= '1' && ev.key <= '6') mark(ZONES[+ev.key - 1][0]);
  else if (ev.key === '0' || ev.key === 'Backspace' || ev.key === 'Delete') mark(null);
  else if (ev.key.toLowerCase() === 'n') { ev.preventDefault(); $('note').focus(); }
});
$('coder').onchange = ev => { coder = ev.target.value; load(); render(); };
function csvCell(v) { const s = String(v ?? ''); return /[",\n]/.test(s) ? '"' + s.replace(/"/g, '""') + '"' : s; }
$('export').onclick = () => {
  if (!coder) { alert('Choose a coder first.'); return; }
  const rows = [['msgid2', 'line_index', 'manual', 'note', 'start_mark'].join(',')];
  emails.forEach(e => { const l = labels(e), m = marks[e.msgid2] || {};
    e.lines.forEach((_, i) => rows.push([e.msgid2, i, l[i] || '', i === 0 ? csvCell(notes[e.msgid2]) : '', m[i] !== undefined ? 1 : 0].join(','))); });
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([rows.join('\n') + '\n'], {type: 'text/csv'}));
  a.download = `__PREFIX___${coder}.csv`; a.click();
};
function parseCSV(text) {
  const out = []; let row = [], cell = '', q = false;
  for (let i = 0; i < text.length; i++) { const ch = text[i];
    if (q) { if (ch === '"' && text[i + 1] === '"') { cell += '"'; i++; } else if (ch === '"') q = false; else cell += ch; }
    else if (ch === '"') q = true; else if (ch === ',') { row.push(cell); cell = ''; }
    else if (ch === '\n') { row.push(cell); out.push(row); row = []; cell = ''; } else if (ch !== '\r') cell += ch; }
  if (cell || row.length) { row.push(cell); out.push(row); } return out;
}
$('import').onclick = () => { if (!coder) { alert('Choose a coder first.'); return; } $('file').click(); };
$('file').onchange = async ev => {
  const f = ev.target.files[0]; if (!f) return;
  const [h, ...body] = parseCSV(await f.text());
  const iM = h.indexOf('msgid2'), iL = h.indexOf('line_index'), iZ = h.indexOf('manual'), iN = h.indexOf('note'), iS = h.indexOf('start_mark');
  marks = {}; notes = {};
  body.forEach(r => { if (r.length < 3) return; const id = r[iM], i = +r[iL];
    if (r[iS] === '1' && r[iZ]) { marks[id] = marks[id] || {}; marks[id][i] = r[iZ]; }
    if (r[iN]) notes[id] = r[iN]; });
  save(); render(); ev.target.value = '';
};
render();
</script></body></html>
'''
data_json = json.dumps(emails, ensure_ascii=False).replace('</', '<\\/')
with open(tool_path, 'w', encoding='utf-8') as f:
    f.write(zone_html.replace('__PREFIX__', prefix).replace('__DATA__', data_json))
print(f"Labelling tool saved to: {tool_path}")

# ---- Compare manual zones with Chipmunk and corpus.ipynb block 5 (runs once a coder's export is in zack_dir) ----
def show(df):
    print(df.to_string(max_colwidth=90))

coder_files = {c: os.path.join(zack_dir, f'{prefix}_{c}.csv') for c in ('coder1', 'coder2')}
coder_files = {c: p for c, p in coder_files.items() if os.path.exists(p)}
labelled = {}
if not coder_files:
    print(f"\nNo coder exports yet: label the emails, export, and save {prefix}_coder1.csv in", zack_dir)
else:
    ref = sample.drop(columns='manual').copy()
    ref['msgid2'] = ref['msgid2'].astype(str)
    ref = ref.sort_values(['msgid2', 'start'])
    ref['line_index'] = ref.groupby('msgid2').cumcount()
    zz = z[['msgid2', 'start', 'chipmunk_mapped', 'zone']].copy()
    zz['msgid2'] = zz['msgid2'].astype(str)
    ref = ref.merge(zz, on=['msgid2', 'start'], how='left')
    ref = ref[ref['text'].str.strip() != '']
    for c, p in coder_files.items():
        lab = pd.read_csv(p, dtype={'msgid2': str})
        lab['manual'] = lab['manual'].fillna('')
        d = ref.merge(lab[['msgid2', 'line_index', 'manual']], on=['msgid2', 'line_index'], how='left')
        d['manual'] = d['manual'].fillna('')
        labelled[c] = d
        done = d[d['manual'] != '']
        print(f"\n===== {c}: {done['msgid2'].nunique()} emails, {len(done):,} of {len(d):,} non-blank lines labelled =====")
        if done.empty:
            continue
        done = done.assign(chipmunk_ok=done['chipmunk_mapped'] == done['manual'],
                           final_ok=done['zone'] == done['manual'])
        print(f"Agreement with manual labels: Chipmunk {done['chipmunk_ok'].mean():.3f} | corpus.ipynb block 5 final {done['final_ok'].mean():.3f}")
        print("\nAgreement per manual zone (share of the zone's lines each method labels the same):")
        show(done.groupby('manual').agg(lines=('manual', 'size'), chipmunk=('chipmunk_ok', 'mean'),
                                        final=('final_ok', 'mean')).round(3))
        print("\nAgreement per stratum:")
        show(done.groupby('stratum').agg(emails=('msgid2', 'nunique'), lines=('manual', 'size'),
                                         chipmunk=('chipmunk_ok', 'mean'), final=('final_ok', 'mean')).round(3))
        print("\nManual zone (rows) by corpus.ipynb block 5 final zone (columns), line counts:")
        show(pd.crosstab(done['manual'], done['zone'], margins=True))
        print("\nManual zone (rows) by Chipmunk zone (columns), line counts:")
        show(pd.crosstab(done['manual'], done['chipmunk_mapped'], margins=True))
        print("\nLines where corpus.ipynb block 5's final zone differs from the manual label (up to 10 per combination):")
        for (m, f), g in done[~done['final_ok']].groupby(['manual', 'zone']):
            print(f"\nmanual: {m} | corpus.ipynb block 5: {f} ({len(g)} lines)")
            show(g.sample(min(len(g), 10), random_state=seed)[['msgid2', 'stratum', 'chipmunk_mapped', 'text']])
    if len(labelled) == 2:
        both = labelled['coder1'].merge(labelled['coder2'][['msgid2', 'line_index', 'manual']],
                                        on=['msgid2', 'line_index'], suffixes=('_1', '_2'))
        both = both[(both['manual_1'] != '') & (both['manual_2'] != '')]
        po = (both['manual_1'] == both['manual_2']).mean()
        p1 = both['manual_1'].value_counts(normalize=True)
        p2 = both['manual_2'].value_counts(normalize=True)
        pe = (p1 * p2.reindex(p1.index).fillna(0)).sum()
        print(f"\nCoder agreement on {len(both):,} lines: {po:.3f} (Cohen's kappa {(po - pe) / (1 - pe):.3f})")

# ---- Test round: precision of corpus.ipynb block 5's authored zone, and the stopping rule ----
# Stop when the lower end of the 95% interval is at least 95%. The interval is the lower of an exact
# binomial interval on lines and a bootstrap over emails (which allows for errors clustering in an email).
if test_round and 'coder1' in labelled:
    import math
    import numpy as np

    def cp_lower(k, n, alpha=0.025):
        """Exact (Clopper-Pearson) lower bound: the p at which P(X >= k | n, p) = alpha."""
        if k == 0:
            return 0.0
        def tail(p):
            return sum(math.exp(math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1)
                                + i * math.log(p) + (n - i) * math.log1p(-p)) for i in range(k, n + 1))
        lo, hi = 0.0, 1.0
        for _ in range(100):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if tail(mid) < alpha else (lo, mid)
        return lo
    t = labelled['coder1']
    t = t[t['manual'] != '']
    a = t[t['zone'] == 'authored']
    k, n = int((a['manual'] == 'authored').sum()), len(a)
    cp_low = cp_lower(k, n)
    per = a.groupby('msgid2')['manual'].agg(lambda s: ((s == 'authored').sum(), len(s)))
    hits = np.array([v[0] for v in per]); tots = np.array([v[1] for v in per])
    rng = np.random.default_rng(seed)
    boots = [hits[i].sum() / tots[i].sum() for i in (rng.integers(0, len(per), len(per)) for _ in range(10000))]
    bs_low = float(np.percentile(boots, 2.5))
    recall_den = int((t['manual'] == 'authored').sum())
    print(f"\nTest round: {t['msgid2'].nunique()} emails, {len(t):,} labelled lines")
    print(f"Authored precision: {k}/{n} = {k / n:.3f} | 95% lower bound: exact {cp_low:.3f}, bootstrap {bs_low:.3f}")
    print(f"Authored recall: {k}/{recall_den} = {k / recall_den:.3f}")
    low = min(cp_low, bs_low)
    print("Stopping rule:", "PASS, the lower bound is at least 95%; stop here." if low >= 0.95
          else f"NOT YET, the lower bound is {low:.3f}; set n_test = {n_test + 20}, rerun, and label the new emails.")